# 10 - look at the results in FiftyOne (per-image results, no GPU)

**CPU runtime is enough.** Nothing is trained or scored again: this reads the
per-image result files the earlier notebooks wrote to Drive and shows every
scored SAR image (SI and the test glaciers COL and Mapple) with its true zones,
its true front, and each run's per-image front error (`mde_m`), whether a front
was found (`has_front`) and per-image zone IoU (`mIoU`).

FiftyOne's anonymous usage tracking is switched off before it is imported.
Nothing is pushed to any hub: the last cell saves the dataset to your Drive as one
zip, and `scripts/view_results.py` opens it on your PC.

In [1]:
# --- cell: install FiftyOne -- run once per runtime, then RESTART THE KERNEL ---
# The install upgrades Pillow; the running kernel still holds the old one, so
# importing FiftyOne now would mix both. After the restart, continue with the
# mount cell below (skip this cell).
import os, subprocess, sys
os.environ["FIFTYONE_DO_NOT_TRACK"] = "true"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fiftyone"], check=True)
print("installed -- now restart the kernel, then run from the next cell")

installed -- now restart the kernel, then run from the next cell


In [2]:
# --- cell: mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [4]:
# --- cell: FiftyOne (tracking off), data ---
import os
from pathlib import Path
os.environ["FIFTYONE_DO_NOT_TRACK"] = "true"          # before the first import of fiftyone
import fiftyone as fo
from fiftyone import ViewField as F
from sartransfer.data import inventory as inv

print("fiftyone", fo.__version__, "| tracking off:", fo.config.do_not_track)
DRIVE = Path("/content/drive/MyDrive")
RAW, RES = DRIVE / "sar-transfer" / "raw", DRIVE / "sar-transfer" / "results"
inv.unzip_caffe(inv.fetch_caffe_zip(RAW), Path("/content/caffe"))
DATA = inv.find_data_root(Path("/content/caffe"))

fiftyone 1.22.0 | tracking off: True
already present: /content/drive/MyDrive/sar-transfer/raw/data_raw.zip  (2.86 GB)
copying zip to local disk (attempt 1) ...
extracted to /content/caffe


In [6]:
# --- cell: build the dataset from the result CSVs on Drive (~5 min) ---
# The run names printed below are the per-run fields in the viewer on your PC (scripts/view_results.py).
from sartransfer.viz import build_dataset, read_results

ds = build_dataset(DATA, RES, Path("/content/fiftyone_media"))
RUNS = sorted(read_results(RES))
print()
print("run fields (use these names below):")
for k in RUNS:
    print(" ", k)

61 result files, 243 scored images
 100% |█████████████████| 243/243 [2.5s elapsed, 0s remaining, 96.2 samples/s]          


INFO:eta.core.utils: 100% |█████████████████| 243/243 [2.5s elapsed, 0s remaining, 96.2 samples/s]          


dataset 'sar-transfer-results': 243 samples, 61 runs

run fields (use these names below):
  TEST_fronts__cradio_v4_h__caffe__tile512
  TEST_fronts__dinov3_l_photo__caffe__tile512
  TEST_fronts__dinov3_l_sat__caffe__tile512
  TEST_fronts__saratrx_v1__unit__tile512
  TEST_fronts__terramind_v1_base_s1__caffe__tile512
  TEST_fronts__unet_scratch_48k__caffe__tile256__na4
  cond_val__cradio_v4_h__caffe__tile512__L32__film__s0
  cond_val__cradio_v4_h__caffe__tile512__L32__film__s1
  cond_val__cradio_v4_h__caffe__tile512__L32__film__s2
  cond_val__cradio_v4_h__caffe__tile512__L32__plain__s0
  cond_val__cradio_v4_h__caffe__tile512__L32__plain__s1
  cond_val__cradio_v4_h__caffe__tile512__L32__plain__s2
  cond_val__dinov3_l_sat__caffe__tile512__L21__film__s0
  cond_val__dinov3_l_sat__caffe__tile512__L21__film__s1
  cond_val__dinov3_l_sat__caffe__tile512__L21__film__s2
  cond_val__dinov3_l_sat__caffe__tile512__L21__plain__s0
  cond_val__dinov3_l_sat__caffe__tile512__L21__plain__s1
  cond_val__dino

In [8]:
# --- cell: save the dataset to Drive as ONE zip, for the viewer on your PC ---
# The viewer does not show through the VS Code Colab connection (tried 2026-09-27),
# so it runs on your PC instead: scripts/view_results.py. Zipped locally first and
# copied once, because many small files written to Drive are slow.
import shutil
EXP = Path("/content/fo_export")
shutil.rmtree(EXP, ignore_errors=True)
ds.export(export_dir=str(EXP), dataset_type=fo.types.FiftyOneDataset, export_media=True)
zip_path = Path(shutil.make_archive("/content/sar-transfer-results", "zip", EXP))
out = DRIVE / "sar-transfer" / "fiftyone"
out.mkdir(parents=True, exist_ok=True)
shutil.copy2(zip_path, out / zip_path.name)
print(f"saved {out / zip_path.name} ({zip_path.stat().st_size / 1e9:.2f} GB)")
print(f"fiftyone {fo.__version__} -- install this same version on your PC")

Exporting samples...


INFO:fiftyone.utils.data.exporters:Exporting samples...


 100% |████████████████████| 243/243 [5.1s elapsed, 0s remaining, 56.4 docs/s]       


INFO:eta.core.utils: 100% |████████████████████| 243/243 [5.1s elapsed, 0s remaining, 56.4 docs/s]       


saved /content/drive/MyDrive/sar-transfer/fiftyone/sar-transfer-results.zip (0.71 GB)
fiftyone 1.22.0 -- install this same version on your PC
